In [22]:
# IMPORTS
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
import joblib
import matplotlib.pyplot as plt 

In [9]:
# raw data
cdcd_data = pd.read_csv("../data/cdcd.csv")
cdcd_data['gender'] = cdcd_data['gender'].map({'Male': 0, 'Female': 1})
cdcd_data = cdcd_data.dropna(subset=['gender']) # drop any where there is no gender value

# only get the columns that will be used: gender, age, bmi, glucose, hypertension, hba1c level, outcome
cdcd_data = cdcd_data[['gender','age','bmi','blood_glucose_level', 'hypertension', 'hbA1c_level', 'diabetes']]
print(cdcd_data['gender'].unique())
cdcd_data.describe()

[1. 0.]


,gender,age,bmi,blood_glucose_level,hypertension,hbA1c_level,diabetes
count,99982.000000,99982.000000,99982.000000,99982.000000,99982.000000,99982.000000,99982.000000
mean,0.585625,41.888076,27.320757,138.057810,0.074863,5.527529,0.085015
std,0.492616,22.517206,6.636853,40.709469,0.263172,1.070665,0.278906
min,0.000000,0.080000,10.010000,80.000000,0.000000,3.500000,0.000000
25%,0.000000,24.000000,23.630000,100.000000,0.000000,4.800000,0.000000
50%,1.000000,43.000000,27.320000,140.000000,0.000000,5.800000,0.000000
75%,1.000000,60.000000,29.580000,159.000000,0.000000,6.200000,0.000000
max,1.000000,80.000000,95.690000,300.000000,1.000000,9.000000,1.000000


In [14]:
cdcd_data = cdcd_data.drop_duplicates()
print("Duplicates: (cdcd)", cdcd_data.duplicated().sum()) 

print(cdcd_data.isnull().sum())       # counts of missing values
print((cdcd_data == 0).sum())         # counts of zeros


Duplicates: (cdcd) 0
gender                 0
age                    0
bmi                    0
blood_glucose_level    0
hypertension           0
hbA1c_level            0
diabetes               0
dtype: int64
gender                 28505
age                        0
bmi                        0
blood_glucose_level        0
hypertension           65480
hbA1c_level                0
diabetes               66039
dtype: int64


In [ ]:
# clean data
# limit cdcd data
cdcd_data = cdcd_data[cdcd_data['age'] >= 18] # adults only
cdcd_data = cdcd_data[(cdcd_data['bmi'] >= 15) & (cdcd_data['bmi'] <= 60)] # bmi should be 15-60
cdcd_data = cdcd_data[(cdcd_data['blood_glucose_level'] >= 50) & (cdcd_data['blood_glucose_level'] <= 250)] # glucose level should be 50-250 

# add interactinon features (experiment)
#cdcd_data['bmi * glucose'] = cdcd_data['bmi'] * cdcd_data['blood_glucose_level']
#cdcd_data['glucose * hbA1c'] = cdcd_data['blood_glucose_level'] * cdcd_data['hbA1c_level']

# resample
majority = cdcd_data[cdcd_data['diabetes'] == 0]  # more with no diabetes
minority = cdcd_data[cdcd_data['diabetes'] == 1]
majority_downsample = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)
cdcd_data = pd.concat([majority_downsample, minority])
print(cdcd_data.describe())
cdcd_data.to_csv("../data/cdcd_clean.csv", index=False)



             gender           age           bmi  blood_glucose_level  \
count  12614.000000  12614.000000  12614.000000         12614.000000   
mean       0.571825     54.318059     30.112641           149.813144   
std        0.494834     17.387356      6.882928            39.246620   
min        0.000000     18.000000     15.040000            80.000000   
25%        0.000000     42.000000     26.290000           130.000000   
50%        1.000000     56.000000     27.725000           145.000000   
75%        1.000000     68.000000     33.470000           160.000000   
max        1.000000     80.000000     59.990000           240.000000   

       hypertension   hbA1c_level     diabetes  bmi * glucose  glucose * hbA1c  
count  12614.000000  12614.000000  12614.00000   12614.000000     12614.000000  
mean       0.164341      6.156128      0.50000    4539.090863       935.259751  
std        0.370600      1.290898      0.50002    1668.328452       351.594659  
min        0.000000      3.

In [37]:
# data
X = cdcd_data[['gender','age','bmi','blood_glucose_level', 'hypertension', 'hbA1c_level']] # features
y = cdcd_data["diabetes"] # target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


print("X_train Shape:",  X_train.shape)
print("X_test Shape:", X_test.shape)
print("Y_train Shape:", y_train.shape)
print("Y_test Shape:", y_test.shape)

X_train Shape: (8829, 6)
X_test Shape: (3785, 6)
Y_train Shape: (8829,)
Y_test Shape: (3785,)


In [33]:
# train logistic regression
log_reg = LogisticRegression(max_iter=1000, C=0.01, class_weight='balanced')
#scores = cross_val_score(log_reg, X, y, cv=5, scoring='accuracy')
log_reg.fit(X_train, y_train)
#print(scores)
#print(scores.mean())
# test logistic regression
y_pred_lr = log_reg.predict(X_test)
lr_acc = accuracy_score(y_test, y_pred_lr)

In [34]:
# train random forest
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# test random forest
y_pred_rf = rf.predict(X_test)
rf_acc = accuracy_score(y_test, y_pred_rf)


In [20]:
# save models
joblib.dump(log_reg, '../models/logreg_model.joblib')
joblib.dump(rf, '../models/rf_model.joblib')

['../models/rf_model.joblib']

In [36]:
# evaluation
# confusion matrix: tn, fp, fn, tp
print("logistic regression accuracy:", lr_acc)
# accuracy: 0.912
cm_lr = confusion_matrix(y_test, y_pred_lr)
print(cm_lr)

print("random forest accuracy", rf_acc)
# accuracy: 0.901
cm_rf = confusion_matrix(y_test, y_pred_rf)
print(cm_rf)

logistic regression accuracy: 0.8589167767503303
[[1653  278]
 [ 256 1598]]
random forest accuracy 0.873447820343461
[[1682  249]
 [ 230 1624]]
